# RQ1 — redundância dentro dos polígonos de especialistas (diagnóstico)

**Motivação.** `effort_reduction_curve.ipynb` mostrou Random Forest e
Logistic Regression praticamente planos do 0% ao 90% de dados de
especialistas; `forest_only_background_check.ipynb` confirmou que não
é efeito de um fundo fácil. Uma explicação alternativa ainda não
testada: a unidade de mistura da curva é o **polígono**, mas o
*treino* continua a usar **todos os píxeis** de cada polígono incluído
— se os píxeis de um mesmo polígono forem quase idênticos no espaço de
embeddings (mesma mancha, mesma assinatura espectral-temporal ao longo
do ano), incluir "mais polígonos" pode não trazer tanta informação
nova quanto a contagem de píxeis sugere.

**O que este notebook faz (só diagnóstico, não re-treina nada ainda):**
1. Distribuição do número de píxeis por polígono — quão concentrada é
   a contagem total de píxeis nalguns polígonos grandes.
2. Decomposição de variância (estilo ANOVA) dos embeddings: que fração
   da variância total é *entre* polígonos vs. *dentro* de cada
   polígono — com um **teste de permutação** (baralhar a atribuição
   polígono→píxel, mantendo os tamanhos dos grupos) para saber se a
   homogeneidade dentro de cada polígono é maior do que seria por
   puro acaso, dado o mesmo conjunto de píxeis.
3. Similaridade de cosseno média, como número mais intuitivo:
   dentro-do-mesmo-polígono vs. entre-polígonos-diferentes vs. entre
   pares de observações de ciência cidadã (referência de "observações
   genuinamente distintas").

Se a redundância for grande, explica o achado "plano" da curva; se
não for, a causa está noutro lado.


## 1. Carregar embeddings e presenças (igual a `effort_reduction_curve.ipynb`)


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import load_pilot_embeddings, match_to_pixel

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)

dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
cs_x, cs_y, cs_emb, cs_n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)

expert_gdf = gpd.read_file(ROOT / "data" / "interim" / "expert_reference" / "expert_polygons_clean.gpkg").to_crs(32629)
pilot_points = gpd.GeoDataFrame(pilot_df, geometry=gpd.points_from_xy(pilot_df["x"], pilot_df["y"]), crs=32629)
joined = gpd.sjoin(pilot_points, expert_gdf[["geometry"]], predicate="within", how="inner")
joined = joined[~joined.index.duplicated(keep="first")]
expert_idx = joined.index.values
expert_poly_id_all = joined["index_right"].values
expert_emb_all = pilot_emb[expert_idx]
valid = ~np.isnan(expert_emb_all).any(axis=1)
expert_emb = expert_emb_all[valid]
expert_poly_id = expert_poly_id_all[valid]

print(f"citizen-science presences: {len(cs_emb):,} (matched {cs_n_matched}/{len(dealbata)})")
print(f"expert presence pixels: {len(expert_emb):,}, across {len(np.unique(expert_poly_id)):,} polygons")
print(f"embedding dimensionality: {expert_emb.shape[1]}")


## 2. Distribuição de píxeis por polígono


In [ ]:
poly_sizes = pd.Series(expert_poly_id).value_counts()
print(poly_sizes.describe())
print()
for k in [1, 2, 5, 10, 20]:
    frac_polys = (poly_sizes >= k).mean()
    frac_px = poly_sizes[poly_sizes >= k].sum() / poly_sizes.sum()
    print(f">= {k:>2} pixels: {frac_polys:.1%} of polygons, holding {frac_px:.1%} of all expert pixels")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(poly_sizes.values, bins=np.arange(1, poly_sizes.max() + 2) - 0.5, color="#2a78d6", edgecolor="white")
ax.set_xlabel("pixels per polygon")
ax.set_ylabel("number of polygons")
ax.set_yscale("log")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Expert polygon size distribution (pilot extent, classifica = 1)")
fig.tight_layout()
plt.show()


## 3. Decomposição de variância entre/dentro de polígonos, com teste de permutação

Para cada dimensão do embedding, `ss_between` é a variância explicada
pela identidade do polígono (quão diferente é a média de cada polígono
da média global) e `ss_within` é a variância residual dentro de cada
polígono. `icc = ss_between / (ss_between + ss_within)` seria 1 se
todos os píxeis de um polígono fossem idênticos entre si (redundância
total) e 0 se a identidade do polígono não explicasse nada.

**O número que importa não é o `icc` sozinho** — alguma homogeneidade
dentro do polígono é esperada mesmo ao acaso, só porque todos os
píxeis vêm de acácia. O que decide a hipótese é comparar o `icc` real
contra um **modelo nulo**: baralhar que píxel pertence a que polígono
(mantendo os tamanhos dos grupos fixos) 200 vezes, e ver se o `icc`
real é claramente maior do que o `icc` de agrupamentos aleatórios do
mesmo conjunto de píxeis. Se for, a redundância é um efeito real da
identidade do polígono (mesma mancha), não só homogeneidade geral da
espécie.


In [ ]:
unique_polys, inverse, counts = np.unique(expert_poly_id, return_inverse=True, return_counts=True)
global_mean = expert_emb.mean(axis=0)


def icc(emb, inverse, counts, global_mean):
    group_sums = np.zeros((len(counts), emb.shape[1]))
    np.add.at(group_sums, inverse, emb)
    group_means = group_sums / counts[:, None]
    ss_within = ((emb - group_means[inverse]) ** 2).sum()
    ss_between = (counts[:, None] * (group_means - global_mean) ** 2).sum()
    return ss_between / (ss_between + ss_within), ss_between, ss_within


icc_obs, ss_between_obs, ss_within_obs = icc(expert_emb, inverse, counts, global_mean)
ss_total = ((expert_emb - global_mean) ** 2).sum()
print(f"ss_between + ss_within = {ss_between_obs + ss_within_obs:,.1f}, ss_total = {ss_total:,.1f} (should match)")
print(f"observed ICC (fraction of variance between real polygons): {icc_obs:.3f}")

rng = np.random.default_rng(0)
n_perm = 200
null_icc = np.empty(n_perm)
for i in range(n_perm):
    shuffled_emb = expert_emb[rng.permutation(len(expert_emb))]
    null_icc[i], _, _ = icc(shuffled_emb, inverse, counts, global_mean)

pctl = (null_icc < icc_obs).mean()
print(f"null ICC (random same-size grouping): mean {null_icc.mean():.3f}, sd {null_icc.std():.3f}, "
      f"95th pctl {np.percentile(null_icc, 95):.3f}")
print(f"observed ICC is above {pctl:.1%} of {n_perm} random-grouping permutations")


## 4. Similaridade de cosseno: dentro do polígono vs. entre polígonos vs. ciência cidadã

Número mais intuitivo que o ICC, para contexto: pares de píxeis
dentro do mesmo polígono deviam ser muito mais parecidos entre si do
que pares de polígonos diferentes — e estes, por sua vez, deviam ser
tão parecidos entre si (ou não) como pares de observações de ciência
cidadã, que já sabemos serem observações genuinamente distintas
(um píxel por presença, sem duplicação — ver `thesis/notes.md`).


In [ ]:
def sample_cosine(emb_a, emb_b, n_pairs, rng):
    norm_a = emb_a / (np.linalg.norm(emb_a, axis=1, keepdims=True) + 1e-12)
    norm_b = emb_b / (np.linalg.norm(emb_b, axis=1, keepdims=True) + 1e-12)
    idx_a = rng.integers(0, len(norm_a), n_pairs)
    idx_b = rng.integers(0, len(norm_b), n_pairs)
    return (norm_a[idx_a] * norm_b[idx_b]).sum(axis=1)


rng = np.random.default_rng(1)
N_PAIRS = 5000

# within-polygon: only polygons with >= 2 pixels
multi = unique_polys[counts >= 2]
within_sims = []
for pid in multi:
    idx = np.where(expert_poly_id == pid)[0]
    k = min(50, len(idx) * (len(idx) - 1) // 2)
    ia = rng.choice(idx, k)
    ib = rng.choice(idx, k)
    keep = ia != ib
    if keep.sum() == 0:
        continue
    norm = expert_emb / (np.linalg.norm(expert_emb, axis=1, keepdims=True) + 1e-12)
    within_sims.append((norm[ia[keep]] * norm[ib[keep]]).sum(axis=1))
within_sims = np.concatenate(within_sims)

# between-polygon: random pairs forced to come from different polygons
idx_a = rng.integers(0, len(expert_emb), N_PAIRS)
idx_b = rng.integers(0, len(expert_emb), N_PAIRS)
keep = expert_poly_id[idx_a] != expert_poly_id[idx_b]
norm_e = expert_emb / (np.linalg.norm(expert_emb, axis=1, keepdims=True) + 1e-12)
between_sims = (norm_e[idx_a[keep]] * norm_e[idx_b[keep]]).sum(axis=1)

# citizen-science: random pairs (already one pixel per presence, no polygon structure)
cs_sims = sample_cosine(cs_emb, cs_emb, N_PAIRS, rng)

summary = pd.DataFrame({
    "pair type": ["within same expert polygon", "between different expert polygons", "between citizen-science presences"],
    "n pairs": [len(within_sims), len(between_sims), len(cs_sims)],
    "mean cosine sim": [within_sims.mean(), between_sims.mean(), cs_sims.mean()],
    "median cosine sim": [np.median(within_sims), np.median(between_sims), np.median(cs_sims)],
    "sd": [within_sims.std(), between_sims.std(), cs_sims.std()],
})
summary.round(4)


## 5. Leitura do diagnóstico (2026-11-07)

**A redundância dentro do polígono é real e estatisticamente muito
forte — mas os polígonos, neste recorte piloto, são pequenos demais
para isso explicar sozinho a curva plana.**

- **Tamanho dos polígonos:** mediana 1 píxel, média 1,9, **máximo 9
  píxeis**. Não há nenhum polígono grande a dominar o conjunto de
  treino — a hipótese de "poucos polígonos enormes, quase-duplicados,
  a afogar o sinal" não se confirma neste recorte. Ainda assim, 63,8%
  de todos os píxeis de especialistas vêm de polígonos com 2+ píxeis.
- **ICC = 0,881, muito acima do nulo por permutação (média 0,525, 95º
  percentil 0,538; o valor real fica acima de 100% das 200
  permutações).** Isto confirma que saber a que polígono um píxel
  pertence explica ~88% da variância total do embedding — muito mais
  do que um agrupamento aleatório do mesmo conjunto de píxeis
  explicaria por puro acaso. A redundância dentro do polígono é um
  efeito real da identidade da mancha, não só homogeneidade geral da
  espécie.
- **Similaridade de cosseno, do mais ao menos parecido:** dentro do
  mesmo polígono 0,921 › entre polígonos diferentes 0,778 › entre
  presenças de ciência cidadã 0,612. O resultado mais inesperado não é
  o primeiro número (esperado) mas o segundo: **polígonos de
  especialistas diferentes entre si (0,778) já são mais parecidos uns
  com os outros do que observações de ciência cidadã o são entre si
  (0,612).** Ou seja, mesmo contando só a diversidade *entre*
  polígonos (ignorando a redundância *dentro* de cada um), os dados de
  especialistas cobrem, aqui, uma gama de condições mais estreita do
  que a ciência cidadã já cobre — possivelmente por estarem
  concentrados numa zona/habitat mais homogéneo do distrito-piloto, não
  só por efeito de polígono.

**Conclusão provisória:** há redundância real dentro do polígono
(confirmada estatisticamente), mas a dimensão pequena dos polígonos
neste recorte limita o quanto isso, sozinho, explica a curva plana —
o efeito de "menos amostras independentes do que a contagem de píxeis
sugere" existe mas é modesto (no máximo ~1457 unidades efetivas em vez
de 2771 píxeis, não uma redução drástica). A observação da baixa
diversidade *entre* polígonos pode ser, afinal, tão ou mais relevante
para explicar a curva plana do que a redundância *dentro* de cada um.

A pista da diversidade *entre* polígonos foi investigada a seguir em
[`expert_polygon_geographic_diversity.ipynb`](expert_polygon_geographic_diversity.ipynb),
que deu a explicação mais forte e direta para a curva plana
(concentração de habitat, não geografia nem redundância de píxeis).

**Fundamentação:** decomposição de variância entre/dentro de grupo
(estilo ANOVA de um fator) e teste de permutação são técnicas
estatísticas padrão; não verifiquei uma citação específica para a sua
aplicação a dados de reflectância/embeddings de sensoriamento remoto
agrupados por polígono — é raciocínio meu a partir dos números desta
experiência.

**Em aberto:**
- Por que razão os polígonos de especialistas são, como grupo, menos
  diversos entre si do que a ciência cidadã (0,778 vs. 0,612) — pode
  ser cobertura geográfica mais concentrada, não falta de valor dos
  dados em si.
- Só testado no recorte-piloto (distrito de Coimbra); pode não
  generalizar a um conjunto de polígonos maior/mais disperso.
- Resultado ainda não confirmado pelo orientador.

**Onde:** este notebook.
